In [ ]:
from pathlib import Path

# Run this baseline for the attached XGLUE Kaggle dataset.
# Add the XGLUE dataset to the Kaggle notebook inputs before running all cells.
DATASET_KEYS = ("xglue4",)
RUN_LABEL = "deckard_baseline"


# XGLUE Deckard Baseline

Kaggle-ready Deckard-style no-train baseline. This is a self-contained characteristic-vector reproduction for XGLUE: it hashes token, token-bigram, and simple structural features, tunes the cosine threshold on valid, and reports test metrics.


In [ ]:
# Executes the unchanged baseline pipeline once per dataset in isolated state.
# This run writes XGLUE-only CSV files, for example xglue4_*_results.csv.
def run_one_dataset(dataset_key: str):
    from pathlib import Path

    DATASET_KEY = dataset_key
    KAGGLE_DATA_ROOT = Path(f"/kaggle/input/datasets/koushamoeini/{DATASET_KEY}")
    WORK_DIR = Path("/kaggle/working")
    SEED = 42

    MAX_TRAIN_PAIRS = None
    MAX_VALID_PAIRS = None
    MAX_TEST_PAIRS = None

    HASH_DIM = 4096
    RESULTS_PATH = WORK_DIR / f"{DATASET_KEY}_deckard_baseline_results.csv"
    SCORES_PATH = WORK_DIR / f"{DATASET_KEY}_deckard_valid_threshold.csv"

    from pathlib import Path
    import gc
    import gzip
    import json
    import math
    import os
    import random
    import re
    import zipfile

    import numpy as np
    import pandas as pd
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    from tqdm.auto import tqdm


    def seed_everything(seed: int = 42):
        random.seed(seed)
        np.random.seed(seed)
        try:
            import torch
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
        except Exception:
            pass


    def is_gzip_file(path: Path) -> bool:
        try:
            with path.open("rb") as f:
                return f.read(2) == b"\x1f\x8b"
        except OSError:
            return False


    def open_text(path: Path):
        return gzip.open(path, "rt", encoding="utf-8") if is_gzip_file(path) else path.open("r", encoding="utf-8")


    def candidate_roots():
        roots = [KAGGLE_DATA_ROOT, Path("/kaggle/input")]
        return [r for r in roots if r.exists()]


    def resolve_file_path(path: Path, *names: str) -> Path:
        if path.is_file():
            return path
        if path.is_dir():
            direct = [path / name for name in names if (path / name).is_file()]
            if direct:
                return direct[0]
            matches = []
            for name in names:
                matches.extend(path.rglob(name))
            matches = [m for m in matches if m.is_file()]
            if matches:
                return sorted(matches, key=lambda p: (len(p.relative_to(path).parts), len(str(p))))[0]
        return path


    def find_file(*names: str) -> Path:
        matches = []
        for root in candidate_roots():
            for name in names:
                direct = root / name
                if direct.is_file():
                    matches.append(direct)
                elif direct.is_dir():
                    resolved = resolve_file_path(direct, *names)
                    if resolved.is_file():
                        matches.append(resolved)
                matches.extend([p for p in root.rglob(name) if p.is_file()])
        if not matches:
            seen = []
            for root in candidate_roots():
                seen.extend(str(p) for p in sorted(root.rglob("*"))[:30])
            raise FileNotFoundError(f"Could not find {names}. First available paths: {seen}")
        return sorted(matches, key=lambda p: (len(p.parts), len(p.name), str(p)))[0]


    def load_pairs(path: Path) -> pd.DataFrame:
        path = resolve_file_path(path, "pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
        compression = "gzip" if is_gzip_file(path) else None
        df = pd.read_csv(path, compression=compression, dtype={"left_id": str, "right_id": str, "split": str, "label": np.int64})
        df["left_id"] = df["left_id"].astype(str)
        df["right_id"] = df["right_id"].astype(str)
        return df


    def load_codes(path: Path) -> dict[str, str]:
        path = resolve_file_path(path, "codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp")
        codes = {}
        with open_text(path) as f:
            for line in tqdm(f, desc="Loading codes", unit="code"):
                if not line.strip():
                    continue
                obj = json.loads(line)
                codes[str(obj["code_id"])] = obj["code"]
        return codes


    def maybe_limit_split(df: pd.DataFrame, split: str, max_rows: int | None, seed: int) -> pd.DataFrame:
        part = df[df["split"] == split].copy()
        if max_rows is not None and len(part) > max_rows:
            part = part.sample(n=max_rows, random_state=seed)
        return part.reset_index(drop=True)


    def metric_dict(labels, scores, threshold: float) -> dict:
        pred = (np.asarray(scores) >= threshold).astype(np.int64)
        labels = np.asarray(labels).astype(np.int64)
        p, r, f1, _ = precision_recall_fscore_support(labels, pred, average="binary", zero_division=0)
        acc = accuracy_score(labels, pred)
        tp = int(((pred == 1) & (labels == 1)).sum())
        fp = int(((pred == 1) & (labels == 0)).sum())
        tn = int(((pred == 0) & (labels == 0)).sum())
        fn = int(((pred == 0) & (labels == 1)).sum())
        return {"P": p, "R": r, "F1": f1, "Acc": acc, "TP": tp, "FP": fp, "TN": tn, "FN": fn}


    def choose_threshold(labels, scores, n_grid: int = 401) -> tuple[float, dict]:
        labels = np.asarray(labels).astype(np.int64)
        scores = np.asarray(scores, dtype=np.float32)
        if len(scores) == 0:
            return 0.5, metric_dict(labels, scores, 0.5)
        qs = np.linspace(0.0, 1.0, n_grid)
        thresholds = np.unique(np.quantile(scores, qs))
        thresholds = np.unique(np.concatenate([thresholds, np.array([0.5], dtype=np.float32)]))
        best_thr = float(thresholds[0])
        best = None
        for thr in thresholds:
            m = metric_dict(labels, scores, float(thr))
            if best is None or (m["F1"], m["Acc"]) > (best["F1"], best["Acc"]):
                best = m
                best_thr = float(thr)
        return best_thr, best


    seed_everything(SEED)
    codes_path = find_file("codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp")
    pairs_path = find_file("pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
    print("codes:", codes_path, "is_file=", codes_path.is_file())
    print("pairs:", pairs_path, "is_file=", pairs_path.is_file())
    codes = load_codes(codes_path)
    pairs_df = load_pairs(pairs_path)
    train_df = maybe_limit_split(pairs_df, "train", MAX_TRAIN_PAIRS, SEED)
    valid_df = maybe_limit_split(pairs_df, "valid", MAX_VALID_PAIRS, SEED + 1)
    test_df = maybe_limit_split(pairs_df, "test", MAX_TEST_PAIRS, SEED + 2)
    print("pairs:")
    print(pairs_df.groupby(["split", "label"]).size())
    print(f"using train/valid/test={len(train_df):,}/{len(valid_df):,}/{len(test_df):,}")


    TOKEN_RE = re.compile(r"[A-Za-z_]\w*|\d+|==|!=|<=|>=|&&|\|\||[{}()\[\];,.:+\-*/%<>=!&|^~?]")
    KEYWORDS = {
        "if", "else", "for", "while", "do", "switch", "case", "return", "break", "continue",
        "try", "catch", "finally", "throw", "throws", "new", "class", "interface", "enum",
        "public", "private", "protected", "static", "final", "void", "int", "long", "float",
        "double", "boolean", "char", "byte", "short", "String"
    }


    def add_hash(vec, key: str, value: float = 1.0):
        idx = hash(key) % HASH_DIM
        vec[idx] += value


    def characteristic_vector(code: str) -> np.ndarray:
        toks = TOKEN_RE.findall(code)
        vec = np.zeros(HASH_DIM, dtype=np.float32)
        for tok in toks:
            norm = "ID" if re.match(r"[A-Za-z_]\w*$", tok) and tok not in KEYWORDS else tok
            norm = "NUM" if tok.isdigit() else norm
            add_hash(vec, "tok:" + norm)
        for a, b in zip(toks, toks[1:]):
            aa = "ID" if re.match(r"[A-Za-z_]\w*$", a) and a not in KEYWORDS else a
            bb = "ID" if re.match(r"[A-Za-z_]\w*$", b) and b not in KEYWORDS else b
            add_hash(vec, "bi:" + aa + "|" + bb, 0.5)
        depth = 0
        max_depth = 0
        for tok in toks:
            if tok in "{([":
                depth += 1
                max_depth = max(max_depth, depth)
            elif tok in "})]":
                depth = max(0, depth - 1)
        scalar = {
            "len_bucket": int(math.log2(len(toks) + 1)),
            "line_bucket": int(math.log2(code.count("\n") + 2)),
            "max_depth": max_depth,
            "returns": code.count("return"),
            "branches": sum(code.count(k) for k in ["if", "switch", "case"]),
            "loops": sum(code.count(k) for k in ["for", "while", "do"]),
        }
        for key, value in scalar.items():
            add_hash(vec, f"{key}:{value}", 2.0)
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec


    needed_ids = set(train_df.left_id) | set(train_df.right_id) | set(valid_df.left_id) | set(valid_df.right_id) | set(test_df.left_id) | set(test_df.right_id)
    code_ids = sorted(needed_ids)
    id_to_row = {cid: i for i, cid in enumerate(code_ids)}
    vectors = np.stack([characteristic_vector(codes.get(cid, "")) for cid in tqdm(code_ids, desc="Deckard vectors")])
    print("codes=", len(code_ids), "dim=", vectors.shape[1])


    def score_pairs(df: pd.DataFrame, batch_size: int = 250_000) -> np.ndarray:
        out = np.empty(len(df), dtype=np.float32)
        left_all = df["left_id"].map(id_to_row).to_numpy(np.int64)
        right_all = df["right_id"].map(id_to_row).to_numpy(np.int64)
        for start in tqdm(range(0, len(df), batch_size), desc="Scoring pairs"):
            end = min(start + batch_size, len(df))
            out[start:end] = (vectors[left_all[start:end]] * vectors[right_all[start:end]]).sum(axis=1)
        return out


    valid_scores = score_pairs(valid_df)
    threshold, valid_metrics = choose_threshold(valid_df["label"].to_numpy(), valid_scores)
    test_scores = score_pairs(test_df)
    test_metrics = metric_dict(test_df["label"].to_numpy(), test_scores, threshold)

    row = {
        "Method": "Deckard",
        "BestEpoch": "",
        "BestValidF1": valid_metrics["F1"],
        **test_metrics,
        "Threshold": threshold,
        "TrainPairs": len(train_df),
        "ValidPairs": len(valid_df),
        "TestPairs": len(test_df),
    }
    pd.DataFrame([row]).to_csv(RESULTS_PATH, index=False)
    pd.DataFrame([{"Threshold": threshold, **valid_metrics}]).to_csv(SCORES_PATH, index=False)
    print(pd.DataFrame([row]))
    print("saved:", RESULTS_PATH)

    if "results_df" in locals():
        return results_df.copy()
    if "result" in locals():
        return pd.DataFrame([result])
    if "row" in locals():
        return pd.DataFrame([row])
    raise RuntimeError("The baseline did not produce a result table.")


from IPython.display import display
import pandas as pd

all_dataset_results = {}
for current_dataset_key in DATASET_KEYS:
    print("\n" + "=" * 96)
    print(f"Running {current_dataset_key.upper()}")
    print("=" * 96)
    dataset_results = run_one_dataset(current_dataset_key)
    dataset_results.insert(0, "Dataset", current_dataset_key.upper())
    all_dataset_results[current_dataset_key] = dataset_results

print("\n" + "=" * 96)
print("Final result tables")
print("=" * 96)
for current_dataset_key in DATASET_KEYS:
    print(f"\n{current_dataset_key.upper()} results")
    display(all_dataset_results[current_dataset_key])

combined_results = pd.concat(
    [all_dataset_results[key] for key in DATASET_KEYS],
    ignore_index=True,
)
combined_path = Path("/kaggle/working") / f"{RUN_LABEL}_combined_dataset_results.csv"
combined_results.to_csv(combined_path, index=False)
print("Combined results:", combined_path)
